In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
from torch.utils.data import DataLoader, Dataset


In [ ]:
train_df = pd.read_csv('sign_mnist_train.csv')
test_df = pd.read_csv('sign_mnist_test.csv')

train_labels = train_df['label'].values
train_images = train_df.drop('label', axis=1).values

test_labels = test_df['label'].values
test_images = test_df.drop('label', axis=1).values


In [ ]:
class SignLanguageDataset(Dataset):

  def __init__(self, images, labels):
    self.images = images
    self.labels = labels

  def __len__(self):
    return len(self.images)

  def __getitem__(self, idx):
    image = self.images[idx]
    image = image.reshape(28, 28)
    image = image / 255.0
    image = torch.tensor(image, dtype=torch.float32).unsqueeze(0)
    label = torch.tensor(self.labels[idx], dtype=torch.long)

    return image, label


In [ ]:
train_dataset = SignLanguageDataset(train_images, train_labels)
test_dataset = SignLanguageDataset(test_images, test_labels)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=True)

In [ ]:
def train_model(model, num_epochs = 20):

  optimizer = optim.Adam(model.parameters(), lr=0.001)
  loss_function = nn.CrossEntropyLoss()

  for epoch in range(num_epochs):
    model.train()

    correct = 0
    total = 0

    for imgs, labels in train_loader:
      preds = model(imgs)
      loss = loss_function(preds, labels)

      optimizer.zero_grad()
      loss.backward()
      optimizer.step()

      predicted = preds.argmax(dim=1)
      correct += (predicted == labels).sum().item()
      total += labels.size(0)

    accuracy = correct / total
    print("Epoch", epoch+1, "Train Accuracy:", accuracy)

    if accuracy == 1.0:
      print("Model has reached 100% accuracy!")
      break



In [ ]:
def test_model(model, num_epochs = 20):

  model.eval()

  correct = 0
  total = 0

  with torch.no_grad():
    for imgs, labels in test_loader:
      outputs = model(imgs)
      predicted = outputs.argmax(dim=1)
      correct += (predicted == labels).sum().item()
      total += labels.size(0)

  accuracy = correct / total
  print("Test Accuracy:", accuracy)
  return accuracy

In [ ]:
def count_params(model):
  return sum(p.numel() for p in model.parameters())

In [ ]:
class BaseCNN(nn.Module):
  def __init__(self):
    super().__init__()

    self.conv1 = nn.Conv2d(1, 8, kernel_size = 3, padding=1)
    self.act1 = nn.Tanh()
    self.pool1 = nn.MaxPool2d(2)

    self.conv2 = nn.Conv2d(8, 16, kernel_size = 3, padding=1)
    self.act2 = nn.Tanh()
    self.pool2 = nn.MaxPool2d(2)

    self.fc1 = nn.Linear(16*7*7, 32)
    self.act3 = nn.Tanh()
    self.fc2 = nn.Linear(32, 25)

  def forward(self, x):
    x = x.view(x.size(0), 1, 28, 28)
    out = self.pool1(self.act1(self.conv1(x)))
    out = self.pool2(self.act2(self.conv2(out)))
    out = out.view(out.size(0), -1)
    out = self.fc2(self.act3(self.fc1(out)))

    return out


In [ ]:
cnn = BaseCNN()
train_model(cnn)
cnn_test_accuracy = test_model(cnn)
print("CNN Parameters: ", count_params(cnn))

Epoch 1 Train Accuracy: 0.5969404480058277
Epoch 2 Train Accuracy: 0.93735203059552
Epoch 3 Train Accuracy: 0.9921325805864142
Epoch 4 Train Accuracy: 0.9989801493352759
Epoch 5 Train Accuracy: 1.0
Model has reached 100% accuracy!
Test Accuracy: 0.8959843837144451
CNN Parameters:  27193


The test accuracy is not that good even though the training accuracy gets pretty high. I'm going to use Dropout to randomly deactivate some neurons to try to prevent overfitting in case there is some overfitting happening.

In [ ]:
class DropoutCNN(nn.Module):
  def __init__(self):
    super().__init__()

    self.conv1 = nn.Conv2d(1, 8, kernel_size=3, padding=1)
    self.act1 = nn.Tanh()
    self.pool1 = nn.MaxPool2d(2)
    self.dropout1 = nn.Dropout2d(0.4)

    self.conv2 = nn.Conv2d(8, 16, kernel_size = 3, padding=1)
    self.act2 = nn.Tanh()
    self.pool2 = nn.MaxPool2d(2)
    self.dropout2 = nn.Dropout2d(0.4)

    self.fc1 = nn.Linear(16*7*7, 32)
    self.act3 = nn.Tanh()
    self.fc2 = nn.Linear(32, 25)

  def forward(self, x):
    x = x.view(x.size(0), 1, 28, 28)
    out = self.pool1(self.act1(self.conv1(x)))
    out = self.dropout1(out)
    out = self.pool2(self.act2(self.conv2(out)))
    out = self.dropout2(out)
    out = out.view(out.size(0), -1)
    out = self.fc2(self.act3(self.fc1(out)))

    return out

In [ ]:
cnn_dropout = DropoutCNN()
train_model(cnn_dropout)
cnn_dropout_test_accuracy = test_model(cnn_dropout)
print("CNN Parameters: ", count_params(cnn_dropout))

Epoch 1 Train Accuracy: 0.3750136587142597
Epoch 2 Train Accuracy: 0.7242396649062102
Epoch 3 Train Accuracy: 0.8383536696412311
Epoch 4 Train Accuracy: 0.8977235476233837
Epoch 5 Train Accuracy: 0.9308686942269168
Epoch 6 Train Accuracy: 0.9501730103806229
Epoch 7 Train Accuracy: 0.9641959570205791
Epoch 8 Train Accuracy: 0.9735567291932252
Epoch 9 Train Accuracy: 0.9779275177563286
Epoch 10 Train Accuracy: 0.9825532689856128
Epoch 11 Train Accuracy: 0.9836095428883628
Epoch 12 Train Accuracy: 0.9852850118375523
Epoch 13 Train Accuracy: 0.9876160990712074
Epoch 14 Train Accuracy: 0.9879074849754144
Epoch 15 Train Accuracy: 0.9889637588781642
Epoch 16 Train Accuracy: 0.9887452194500091
Epoch 17 Train Accuracy: 0.9887816426880349
Epoch 18 Train Accuracy: 0.9904935348752504
Epoch 19 Train Accuracy: 0.9903114186851211
Epoch 20 Train Accuracy: 0.9915498087780004
Test Accuracy: 0.869771332961517
CNN Parameters:  27193


The test accuracy decreased, so there was probably no overfitting happening just underfitting. So I'm going to try to train with ReLU instead of Tanh to try to get a higer accuracy.

In [ ]:
class ReLUCNN(nn.Module):
  def __init__(self):
    super().__init__()

    self.conv1 = nn.Conv2d(1, 8, kernel_size = 3, padding=1)
    self.act1 = nn.ReLU()
    self.pool1 = nn.MaxPool2d(2)

    self.conv2 = nn.Conv2d(8, 16, kernel_size = 3, padding=1)
    self.act2 = nn.ReLU()
    self.pool2 = nn.MaxPool2d(2)

    self.fc1 = nn.Linear(16*7*7, 32)
    self.act3 = nn.ReLU()
    self.fc2 = nn.Linear(32, 25)

  def forward(self, x):
    x = x.view(x.size(0), 1, 28, 28)
    out = self.pool1(self.act1(self.conv1(x)))
    out = self.pool2(self.act2(self.conv2(out)))
    out = out.view(out.size(0), -1)
    out = self.fc2(self.act3(self.fc1(out)))

    return out


In [ ]:
cnn_relu = ReLUCNN()
train_model(cnn_relu, 20)
cnn_test_accuracy = test_model(cnn_relu, 20)
print("CNN Parameters: ", count_params(cnn_relu))

Epoch 1 Train Accuracy: 0.46363139683117827
Epoch 2 Train Accuracy: 0.8226552540520853
Epoch 3 Train Accuracy: 0.9011473319978146
Epoch 4 Train Accuracy: 0.9425241303951921
Epoch 5 Train Accuracy: 0.9660899653979239
Epoch 6 Train Accuracy: 0.9814969950828629
Epoch 7 Train Accuracy: 0.9926425059187762
Epoch 8 Train Accuracy: 0.9954835184847933
Epoch 9 Train Accuracy: 0.9954835184847933
Epoch 10 Train Accuracy: 0.9974503733381898
Epoch 11 Train Accuracy: 0.9964305226734657
Epoch 12 Train Accuracy: 0.9988708796211984
Epoch 13 Train Accuracy: 0.9992715352394828
Epoch 14 Train Accuracy: 0.9961755600072847
Epoch 15 Train Accuracy: 0.9952649790566381
Epoch 16 Train Accuracy: 0.9999271535239482
Epoch 17 Train Accuracy: 1.0
Model has reached 100% accuracy!
Test Accuracy: 0.8943112102621306
CNN Parameters:  27193


Training with ReLU also decreased the test accuracy from the base model. I'm going to try to make the CNN model wider.

In [ ]:
class WideCNN(nn.Module):
  def __init__(self):
    super().__init__()

    self.conv1 = nn.Conv2d(1, 32, kernel_size=3,padding=1)
    self.act1 = nn.Tanh()
    self.pool1 = nn.MaxPool2d(2)

    self.conv2 = nn.Conv2d(32,16, kernel_size=3,padding=1)
    self.act2 = nn.Tanh()
    self.pool2 = nn.MaxPool2d(2)

    self.fc1 = nn.Linear(16*7*7,64)
    self.act3 = nn.Tanh()
    self.fc2 = nn.Linear(64,25)

  def forward(self,x):

    x = x.view(x.size(0), 1, 28, 28)
    out = self.pool1(self.act1(self.conv1(x)))
    out = self.pool2(self.act2(self.conv2(out)))
    out = out.view(out.size(0), -1)
    out = self.fc2(self.act3(self.fc1(out)))

    return out

In [ ]:
cnn_wide = WideCNN()
train_model(cnn_wide)
cnn_test_accuracy = test_model(cnn_wide)
print("CNN Parameters: ", count_params(cnn_wide))

Epoch 1 Train Accuracy: 0.7024949918047715
Epoch 2 Train Accuracy: 0.987470406119104
Epoch 3 Train Accuracy: 0.9998178838098707
Epoch 4 Train Accuracy: 0.9999635767619741
Epoch 5 Train Accuracy: 1.0
Model has reached 100% accuracy!
Test Accuracy: 0.9099274958170663
CNN Parameters:  56809


After trying to make the CNN wider the testing accuracy actually increased, but there were double the amount of parameters used. And by Epoch 4, the testing accuracy also reached 100%. Next, I'm going to try to make the CNN a deeper model to try to see what would happen.

In [ ]:
class DeepCNN(nn.Module):

  def __init__(self):
    super().__init__()

    self.conv1 = nn.Conv2d(1,16,3,padding=1)
    self.act1 = nn.Tanh()

    self.conv2 = nn.Conv2d(16,16,3,padding=1)
    self.act2 = nn.Tanh()

    self.pool1 = nn.MaxPool2d(2)

    self.conv3 = nn.Conv2d(16,8,3,padding=1)
    self.act3 = nn.Tanh()

    self.pool2 = nn.MaxPool2d(2)

    self.fc1 = nn.Linear(8*7*7,32)
    self.fc2 = nn.Linear(32,25)

  def forward(self,x):

    out = x.view(x.size(0), 1, 28, 28)
    out = self.act1(self.conv1(out))
    out = self.act2(self.conv2(out))
    out = self.pool1(out)
    out = self.pool2(self.act3(self.conv3(out)))
    out = out.view(out.size(0), -1)
    out = self.fc2(self.fc1(out))

    return out

In [ ]:
cnn_deep = DeepCNN()
train_model(cnn_deep)
cnn_test_accuracy = test_model(cnn_deep)
print("CNN Parameters: ", count_params(cnn_deep))

Epoch 1 Train Accuracy: 0.5923875432525951
Epoch 2 Train Accuracy: 0.9436896740120196
Epoch 3 Train Accuracy: 0.9940265889637588
Epoch 4 Train Accuracy: 0.9995264979056638
Epoch 5 Train Accuracy: 0.9999635767619741
Epoch 6 Train Accuracy: 1.0
Model has reached 100% accuracy!
Test Accuracy: 0.8984941438929169
CNN Parameters:  17041


Making the CNN deeper did not really help the accuracy from the base CNN model, so maybe I'll focus on making the model wider instead of deeper.

In [ ]:
class WideCNN2(nn.Module):
  def __init__(self):
    super().__init__()

    self.conv1 = nn.Conv2d(1, 64, kernel_size=3,padding=1)
    self.act1 = nn.Tanh()
    self.pool1 = nn.MaxPool2d(2)

    self.conv2 = nn.Conv2d(64,32, kernel_size=3,padding=1)
    self.act2 = nn.Tanh()
    self.pool2 = nn.MaxPool2d(2)

    self.fc1 = nn.Linear(32*7*7,128)
    self.act3 = nn.Tanh()
    self.fc2 = nn.Linear(128,25)

  def forward(self,x):

    x = x.view(x.size(0), 1, 28, 28)
    out = self.pool1(self.act1(self.conv1(x)))
    out = self.pool2(self.act2(self.conv2(out)))
    out = out.view(out.size(0), -1)
    out = self.fc2(self.act3(self.fc1(out)))

    return out

In [ ]:
cnn_wide2 = WideCNN2()
train_model(cnn_wide2)
cnn_test_accuracy = test_model(cnn_wide2)
print("CNN Parameters: ", count_params(cnn_wide2))

Epoch 1 Train Accuracy: 0.7897650701147332
Epoch 2 Train Accuracy: 0.9990894190493534
Epoch 3 Train Accuracy: 0.9999635767619741
Epoch 4 Train Accuracy: 1.0
Model has reached 100% accuracy!
Test Accuracy: 0.9307027328499721
CNN Parameters:  223161


The test accuracy is the highest so far, but it might be because this model is using the most number of parameters.

In [ ]:
class GAPCNN(nn.Module):

  def __init__(self):
    super().__init__()

    self.conv1 = nn.Conv2d(1,8,3,padding=1)
    self.act1 = nn.Tanh()
    self.pool1 = nn.MaxPool2d(2)

    self.conv2 = nn.Conv2d(8,16,3,padding=1)
    self.act2 = nn.Tanh()
    self.pool2 = nn.MaxPool2d(2)

    self.gap = nn.AdaptiveAvgPool2d((3,3))

    self.fc1 = nn.Linear(16*3*3,32)
    self.act3 = nn.Tanh()
    self.fc2 = nn.Linear(32,25)

  def forward(self,x):

    out = x.view(x.size(0),1,28,28)
    out = self.pool1(self.act1(self.conv1(out)))
    out = self.pool2(self.act2(self.conv2(out)))
    out = self.gap(out)
    out = out.view(out.size(0),-1)
    out = self.fc2(self.act3(self.fc1(out)))

    return out

In [ ]:
cnn_gap = GAPCNN()
train_model(cnn_gap, 20)
cnn_gap_test_accuracy = test_model(cnn_gap)
print("CNN Parameters:", count_params(cnn_gap))

Epoch 1 Train Accuracy: 0.215407029684939
Epoch 2 Train Accuracy: 0.5758513931888545
Epoch 3 Train Accuracy: 0.7374977235476233
Epoch 4 Train Accuracy: 0.8175924239664906
Epoch 5 Train Accuracy: 0.8725186669094882
Epoch 6 Train Accuracy: 0.9078127845565471
Epoch 7 Train Accuracy: 0.9364414496448734
Epoch 8 Train Accuracy: 0.955855035512657
Epoch 9 Train Accuracy: 0.9706792933891824
Epoch 10 Train Accuracy: 0.9810963394645784
Epoch 11 Train Accuracy: 0.9892551447823712
Epoch 12 Train Accuracy: 0.9940630122017847
Epoch 13 Train Accuracy: 0.9964305226734657
Epoch 14 Train Accuracy: 0.9975232198142415
Epoch 15 Train Accuracy: 0.9983609542888363
Epoch 16 Train Accuracy: 0.99894372609725
Epoch 17 Train Accuracy: 0.9990165725733018
Epoch 18 Train Accuracy: 0.9993808049535604
Epoch 19 Train Accuracy: 0.999490074667638
Epoch 20 Train Accuracy: 0.9997814605718448
Test Accuracy: 0.92930842163971
CNN Parameters: 6713


I was able to acheive a similar accuracy to the the wide cnn networks but using way less parameters with this Global Average Pooling Model, which also compressed the model from 16 feature maps of size 7x7 to 3x3 to have less parameters

- do augmentation
- train validation test split (k-fold)
- roc curve
- plots
